# Notebook 4: Combined Module Demo

**PronounceAI — End-to-End Pronunciation Assessment**

This demo runs the full pipeline on a single audio file and displays:
- Module 1 score (word accuracy)
- Module 2 score (acoustic quality)
- Weighted final score
- Combined model score (if trained)
- Feedback

**To use:**
1. Set `AUDIO_PATH` to your audio file
2. Set `REFERENCE_TEXT` to the sentence that was supposed to be spoken
3. Run all cells

## Step 1: Imports and Model Loading

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings('ignore')

MODEL_DIR = Path('../../../').resolve()
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

from combined_module.utils.load_module1 import Module1Scorer
from combined_module.utils.load_module2 import Module2Scorer
from combined_module.utils.scoring import (
    CombinedScorer, compute_final_score, generate_feedback
)

print('Loading models...')
t0 = time.time()
m1 = Module1Scorer()
print(f'  Module 1 (Whisper) loaded in {time.time()-t0:.1f}s')

t0 = time.time()
m2 = Module2Scorer()
print(f'  Module 2 (RF+NN)   loaded in {time.time()-t0:.1f}s')

combined = CombinedScorer()
print(f'  Combined model: {"loaded" if combined.is_available else "not found (run notebooks 1 & 2)"}')

## Step 2: Set Input

Edit the two variables below before running.

In [ ]:
# ── EDIT THESE ─────────────────────────────────────────────────────────────
AUDIO_PATH     = '../../../../librispeech/LibriSpeech/test-clean/1089/134686/1089-134686-0000.flac'
REFERENCE_TEXT = 'he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour-fattened sauce'

# Optional: adjust weights (must sum to 1.0)
W1 = 0.5   # Module 1 weight
W2 = 0.5   # Module 2 weight
# ───────────────────────────────────────────────────────────────────────────

audio_path = Path(AUDIO_PATH)
assert audio_path.exists(), f'Audio file not found: {audio_path}'
duration = librosa.get_duration(path=str(audio_path))
print(f'Audio : {audio_path.name}  ({duration:.2f}s)')
print(f'Ref   : "{REFERENCE_TEXT[:80]}..."' if len(REFERENCE_TEXT) > 80 else f'Ref   : "{REFERENCE_TEXT}"')

## Step 3: Run Full Prediction Pipeline

In [ ]:
print('Scoring...')
t0 = time.time()

transcript    = m1.transcribe(AUDIO_PATH)
from combined_module.utils.load_module1 import _word_f1
module1_score = _word_f1(transcript, REFERENCE_TEXT.strip()) * 100.0
module2_score = m2.score(AUDIO_PATH)
final_score   = compute_final_score(module1_score, module2_score, W1, W2)
feedback      = generate_feedback(module1_score, module2_score, final_score)

combined_score = None
if combined.is_available:
    combined_score = combined.predict(module1_score, module2_score)

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s')

## Step 4: Display Results

In [ ]:
print('\n' + '='*55)
print('          PRONOUNCEAI — ASSESSMENT RESULTS')
print('='*55)
print(f'  Transcript         : {transcript}')
print(f'  Reference          : {REFERENCE_TEXT[:70]}...' if len(REFERENCE_TEXT)>70 else f'  Reference          : {REFERENCE_TEXT}')
print('-'*55)
print(f'  Module 1 Score     : {module1_score:6.1f} / 100  (word accuracy)')
print(f'  Module 2 Score     : {module2_score:6.1f} / 100  (acoustic quality)')
print(f'  Weighted Final     : {final_score:6.1f} / 100  (w1={W1}, w2={W2})')
if combined_score is not None:
    print(f'  Combined Model     : {combined_score:6.1f} / 100  (meta RF)')
else:
    print(f'  Combined Model     : N/A (run notebooks 1 & 2 to train)')
print('-'*55)
print(f'  Feedback           : {feedback}')
print('='*55)

## Step 5: Score Breakdown Chart

In [ ]:
scores   = [module1_score, module2_score, final_score]
labels   = ['Module 1\n(Word Accuracy)', 'Module 2\n(Acoustic Quality)', 'Weighted\nFinal Score']
colors   = ['steelblue', 'mediumseagreen', 'darkorange']

if combined_score is not None:
    scores.append(combined_score)
    labels.append('Combined\nModel Score')
    colors.append('mediumpurple')

fig, ax = plt.subplots(figsize=(len(scores)*2.2, 5))
bars = ax.bar(labels, scores, color=colors, width=0.5, alpha=0.85)

for bar, score in zip(bars, scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        f'{score:.1f}',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

ax.axhline(80, color='green',  linestyle='--', linewidth=1, alpha=0.5, label='Good (80)')
ax.axhline(60, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='Fair (60)')
ax.axhline(50, color='red',    linestyle='--', linewidth=1, alpha=0.5, label='Threshold (50)')
ax.set_ylim(0, 110)
ax.set_ylabel('Score (0–100)')
ax.set_title(f'PronounceAI Score Breakdown\nFeedback: {feedback}')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig(Path('../models') / 'demo_score_breakdown.png', dpi=120)
plt.show()

## Step 6: Acoustic Feature Visualization

In [ ]:
SR = 16_000
y, _ = librosa.load(AUDIO_PATH, sr=SR, mono=True)

fig, axes = plt.subplots(3, 1, figsize=(12, 8))

# MFCC
mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=40)
img  = librosa.display.specshow(mfcc, sr=SR, x_axis='time', ax=axes[0], cmap='viridis')
axes[0].set_title('MFCC (40 coefficients)')
fig.colorbar(img, ax=axes[0])

# Pitch
f0, vf, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
times = librosa.times_like(f0, sr=SR)
axes[1].plot(times, f0, color='steelblue', linewidth=1.5)
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Hz')
axes[1].set_title('Pitch (F0)')

# RMS Energy
rms = librosa.feature.rms(y=y)[0]
rms_times = librosa.times_like(rms, sr=SR)
axes[2].plot(rms_times, rms, color='mediumseagreen', linewidth=1.5)
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('RMS')
axes[2].set_title('Energy (RMS)')

plt.suptitle(f'Acoustic Features — {Path(AUDIO_PATH).name}', fontsize=12)
plt.tight_layout()
plt.savefig(Path('../models') / 'demo_acoustic_features.png', dpi=120)
plt.show()